# 01 — Ingest & Explore

First-run notebook. Scans the entries directory, embeds all new entries, and writes them to LanceDB. After this, the data is ready for analysis notebooks.

In [1]:
import sys
sys.path.insert(0, "../src")

import pandas as pd
from journal.config import ENTRIES_DIR, LANCE_ROOT, STATE_DIR
from journal.embed import OllamaEmbedder
from journal.ingest import Ingestor
from journal.store import Store

LANCE_ROOT.mkdir(parents=True, exist_ok=True)
STATE_DIR.mkdir(parents=True, exist_ok=True)
store = Store(LANCE_ROOT)
store.create_tables()
ingestor = Ingestor(store=store, embedder=OllamaEmbedder(), state_path=STATE_DIR / "seen_files.json")
report = ingestor.scan_and_ingest(ENTRIES_DIR)
print(report)

IngestReport(files_processed=1831, entries_added=3035, files_skipped=0, errors=[])


In [2]:
df = store.entries_to_pandas()
print(f"total entries: {len(df):,}")
print(f"date range: {df['date'].min()} → {df['date'].max()}")
print()
print("entries per year:")
# Convert date column to datetime for grouping
df['date_dt'] = pd.to_datetime(df['date'])
print(df.groupby(df["date_dt"].dt.year).size())

total entries: 3,035
date range: 2020-06-30 → 2026-06-22

entries per year:
date_dt
2020    293
2021    385
2022    812
2023     95
2024    280
2025    713
2026    457
dtype: int64


In [3]:
df.head(3)

,id,file_path,date,timestamp,time_of_day,day_of_week,text,category,embedding,ingested_at,date_dt
0,3e44bbb0c6e40641,/Users/smukherjee/Documents/Archives/Backups/A...,2020-06-30,2020-06-30,night,Tue,Everything they do they are right. I don't eve...,Stress,"[0.03083057, 0.026254091, -0.16664143, -0.0053...",2026-06-30 22:41:48.723740,2020-06-30
1,c2bb094021ff06e6,/Users/smukherjee/Documents/Archives/Backups/A...,2020-06-30,2020-06-30,night,Tue,www.github.com/SubhadityaMukherjee\nhttps://ww...,Notes,"[-0.008916306, 0.08669667, -0.152947, -0.00914...",2026-06-30 22:41:48.723767,2020-06-30
2,cde0ebc2fd75d28a,/Users/smukherjee/Documents/Archives/Backups/A...,2020-06-30,2020-06-30,night,Tue,## Mayu,NaN,"[0.010099525, -0.00038186825, -0.15909086, 0.0...",2026-06-30 22:41:48.723780,2020-06-30
